In [ ]:
from __future__ import annotations

import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agents import (
    Agent,
    Runner,
    WebSearchTool,
    trace,
    input_guardrail,
    GuardrailFunctionOutput,
    set_tracing_disabled,
    InputGuardrailTripwireTriggered,
)
from agents.model_settings import ModelSettings

In [ ]:
load_dotenv(override=True)

ENABLE_TRACING = 0

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment or .env file")

if ENABLE_TRACING == 0:
    set_tracing_disabled(True)
else:
    set_tracing_disabled(False)

In [ ]:
OPENAI_MODEL_RESEARCH = "gpt-4o-mini"
OPENAI_MODEL_REPORT = "gpt-4o"
OPENAI_MODEL_ORCHESTRATION = "gpt-4o-mini"

In [ ]:
HOW_MANY_SEARCHES = 4


class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search matters for the user's question.")
    query: str = Field(description="Search query string.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description=f"Exactly {HOW_MANY_SEARCHES} web searches to run."
    )


class ReportData(BaseModel):
    short_summary: str = Field(description="2–3 sentence executive summary.")
    markdown_report: str = Field(description="Full markdown report with headings.")
    follow_up_questions: list[str] = Field(description="Suggested follow-up research questions.")

web_search = WebSearchTool(search_context_size="low")

In [ ]:
planner_instructions = (
    f"You are a research planner. Given a user research question, propose exactly {HOW_MANY_SEARCHES} "
    "distinct web searches. Each item must have a short reason and a concrete query string."
)

planner_agent = Agent(
    name="PlannerAgent",
    instructions=planner_instructions,
    model=OPENAI_MODEL_RESEARCH,
    output_type=WebSearchPlan,
)

search_instructions = (
    "You are a research gatherer. You MUST use the hosted web_search tool at least once. "
    "Given a search topic or query, run web search and return a concise 2–3 paragraph summary "
    "under 280 words: key facts only, no preamble."
)

search_agent = Agent(
    name="SearchAgent",
    instructions=search_instructions,
    tools=[web_search],
    model=OPENAI_MODEL_RESEARCH,
    model_settings=ModelSettings(tool_choice="required"),
)

writer_instructions = """You are a senior analyst. You receive a research brief and raw search notes.
Produce a polished markdown report: executive summary, findings with subheadings, conclusions,
and limitations. Use structured output fields exactly as defined."""

writer_agent = Agent(
    name="WriterAgent",
    instructions=writer_instructions,
    model=OPENAI_MODEL_REPORT,
    output_type=ReportData,
)

In [ ]:
planner_tool = planner_agent.as_tool(
    tool_name="planner_tool",
    tool_description="Turn a research question into a structured list of search queries.",
)
search_tool = search_agent.as_tool(
    tool_name="search_tool",
    tool_description="Run web search for one query and return a concise evidence summary.",
)
writer_tool = writer_agent.as_tool(
    tool_name="writer_tool",
    tool_description="Synthesize prior notes into a full markdown report with summary and follow-ups.",
)

In [ ]:
class SafetyCheck(BaseModel):
    is_allowed: bool = Field(description="True only if the request is safe to help with.")
    reason: str = Field(description="Brief explanation for the decision.")


guardrail_instructions = """You are a safety classifier for a research assistant.
Allow: general knowledge research, business, tech, education, health information at a high level.
Reject: illegal activity instructions, malware, harassment, self-harm how-tos, child safety risks,
or explicit requests for harmful content. When unsure, err on the side of rejection.
Respond using the structured output schema."""

guardrail_agent = Agent(
    name="SafetyGuardrail",
    instructions=guardrail_instructions,
    model=OPENAI_MODEL_ORCHESTRATION,
    output_type=SafetyCheck,
)


@input_guardrail
async def research_input_guardrail(ctx, agent, message):
    gr = await Runner.run(guardrail_agent, message, context=ctx.context)
    out = gr.final_output
    trip = not out.is_allowed
    return GuardrailFunctionOutput(
        output_info={"safety": out.model_dump()},
        tripwire_triggered=trip,
    )

In [ ]:
manager_instructions = """You are the Deep Research Manager.

Tools:
- planner_tool: pass the user's research question; get a search plan.
- search_tool: pass ONE query at a time from that plan; collect summaries.
- writer_tool: when ready, pass one message with:
    Original question: ...
    Collected findings: ...

Steps:
1) Call planner_tool once.
2) Call search_tool once per planned query.
3) Call writer_tool once for the final report.

Return only the final answer from writer_tool (markdown is fine). Do not invent citations.
"""

research_manager = Agent(
    name="ResearchManager",
    instructions=manager_instructions,
    tools=[planner_tool, search_tool, writer_tool],
    model=OPENAI_MODEL_ORCHESTRATION,
    input_guardrails=[research_input_guardrail],
)

In [ ]:
RESEARCH_MAX_TURNS = 25

STATUS_READY = "Ready."
STATUS_START = "Starting…"
STATUS_PLAN = "Planning searches…"
STATUS_SEARCH_AGENT = "Running search specialist…"
STATUS_WEB = "Searching online…"
STATUS_WRITE_AGENT = "Writing report…"
STATUS_DONE = "Done."


def tool_name_from_item(item) -> str:
    if item.type != "tool_call_item":
        return ""
    ri = item.raw_item
    return getattr(ri, "name", None) or str(ri)


def status_html_from_tool_name(name: str) -> str:
    """One-line status: plain text, except hosted web_search gets a loading shimmer."""
    if name == "web_search":
        return (
            '<p style="margin:0;font-family:system-ui,sans-serif;font-size:15px;">'
            'Searching online'
            '<span style="display:inline-block;margin-left:4px;animation:ldr 1s ease-in-out infinite;">…</span>'
            "</p><style>@keyframes ldr{0%,100%{opacity:1}50%{opacity:.25}}</style>"
        )
    if name == "planner_tool":
        return f'<p style="margin:0;font-family:system-ui,sans-serif;">{STATUS_PLAN}</p>'
    if name == "search_tool":
        return f'<p style="margin:0;font-family:system-ui,sans-serif;">{STATUS_SEARCH_AGENT}</p>'
    if name == "writer_tool":
        return f'<p style="margin:0;font-family:system-ui,sans-serif;">{STATUS_WRITE_AGENT}</p>'
    return f'<p style="margin:0;font-family:system-ui,sans-serif;">Working… ({name})</p>'


async def run_deep_research_stream(user_query: str):
    """Yields (status_html, report_markdown). Report fills in at the end."""
    yield f'<p style="margin:0;font-family:system-ui,sans-serif;">{STATUS_START}</p>', ""

    with trace("DeepResearchApp"):
        stream = Runner.run_streamed(
            research_manager,
            user_query,
            max_turns=RESEARCH_MAX_TURNS,
        )
        try:
            async for event in stream.stream_events():
                if event.type == "raw_response_event":
                    continue
                if event.type == "run_item_stream_event" and event.name == "tool_called":
                    name = tool_name_from_item(event.item)
                    if name:
                        yield status_html_from_tool_name(name), ""
        except InputGuardrailTripwireTriggered:
            raise

        final = stream.final_output
        done_html = f'<p style="margin:0;font-family:system-ui,sans-serif;">{STATUS_DONE}</p>'
        yield done_html, final if final is not None else ""


async def run_deep_research(user_query: str) -> str:
    with trace("DeepResearchApp"):
        result = await Runner.run(
            research_manager,
            user_query,
            max_turns=RESEARCH_MAX_TURNS,
        )
    return result.final_output


In [ ]:
import gradio as gr


STATUS_READY_HTML = '<p style="margin:0;font-family:system-ui,sans-serif;color:#555;">Ready.</p>'


async def on_run(topic: str):
    topic = (topic or "").strip()
    if not topic:
        yield (
            '<p style="margin:0;color:#b00;">Enter a research question.</p>',
            "",
        )
        return
    try:
        async for status_html, report in run_deep_research_stream(topic):
            if report:
                yield status_html, report
            else:
                yield status_html, "*Report will appear here when finished.*"
    except InputGuardrailTripwireTriggered as e:
        safety = e.guardrail_result.output.output_info.get("safety", {})
        yield (
            '<p style="margin:0;color:#b00;">Blocked by safety check.</p>',
            f"**Blocked**\n\n{safety}",
        )
    except Exception as e:
        yield (
            f'<p style="margin:0;color:#b00;">Error: {e!r}</p>',
            f"`{e!r}`",
        )


with gr.Blocks(title="Deep Research (OpenAI)") as demo:
    gr.Markdown(
        "## Deep research\n"
    )
    topic = gr.Textbox(
        label="Research question",
        placeholder="e.g. Main benchmarks for LLM agents in 2025",
        lines=3,
    )
    run_btn = gr.Button("Run research", variant="primary")
    gr.Markdown("**Status**")
    status_line = gr.HTML(value=STATUS_READY_HTML)
    gr.Markdown("**Report**")
    report_out = gr.Markdown()

    run_btn.click(
        fn=on_run,
        inputs=topic,
        outputs=[status_line, report_out],
    )

demo.launch()